In [ ]:
import pandas as pd
import numpy as np

In [ ]:
measure = "MEP Latency (msec)"

In [ ]:
cohort_df = pd.read_csv("data/demographics.csv")
cohort_df.head()

In [ ]:
cohort_df[["age", "height", "weight"]].agg(["mean", "std", "min", "max"]).T.to_csv(
    "results/cohort_summary/continuous.csv"
)

In [ ]:
cohort_df["count"] = 1
cohort_df.groupby("sex")["count"].sum()

sex_summary_df = pd.DataFrame(index=cohort_df["sex"].unique())
sex_summary_df = sex_summary_df.join(cohort_df.groupby("sex")["count"].sum())
sex_summary_df["percentage"] = sex_summary_df["count"] / len(cohort_df)
sex_summary_df.to_csv("results/cohort_summary/sex.csv")

In [ ]:
cohort_df[["age", "height", "weight"]].corr().round(2).to_csv(
    "results/cohort_summary/corr.csv"
)

In [ ]:
# Measure specific data descriptions
df = pd.read_csv(f"data/{measure}.csv")

In [ ]:
df.groupby(["muscle", "limb", "side"])[["value"]].quantile(
    [0, 0.25, 0.5, 0.75, 1]
).rename_axis(["muscle", "limb", "side", "quantile"]).reset_index().pivot(
    index=["muscle", "limb", "quantile"], columns="side", values="value"
).reset_index(
    level=2
).to_csv(
    f"results/{measure}/distribution_description/summary_by_muscle.csv", index=True
)

In [ ]:
df.groupby(["limb", "sex", "side"])[["value"]].quantile(
    [0, 0.25, 0.5, 0.75, 1]
).rename_axis(["limb", "sex", "side", "quantile"]).reset_index().pivot(
    index=["limb", "sex", "quantile"], columns="side", values="value"
).to_csv(
    f"results/{measure}/distribution_description/summary_by_limb_and_sex.csv",
    index=True,
)

In [ ]:
# Asymmetry (right - left)

asymmetry_df = df.pivot(
    index=["patient_id", "muscle"], columns="side", values="value"
).reset_index()
asymmetry_df["r_minus_l"] = asymmetry_df["Right"] - asymmetry_df["Left"]
asymmetry_df["r_minus_l_abs"] = np.abs(asymmetry_df["Right"] - asymmetry_df["Left"])

In [ ]:
asymmetry_df.isna().sum()  # Missing vals. due to patients missing one side measure

In [ ]:
# drop missing pairs and store data
asymmetry_df.dropna(subset=["r_minus_l"]).to_csv(
    f"data/{measure}_asymmetry.csv", index=False
)

In [ ]:
asymmetry_df.groupby("muscle")[["r_minus_l"]].quantile(
    [0, 0.25, 0.5, 0.75, 1]
).rename_axis(["muscle", "quantile"]).to_csv(
    f"results/{measure}/distribution_description/asymmetry_by_muscle.csv"
)